# TFM Deteccion - Prueba mini e2e

Objetivo: validar el pipeline completo (train -> checkpoint -> eval) en Colab con GPU usando un subconjunto reducido.

Tamanos: 2 categorias de progan_train + progan_val completo. 4 modelos x (2 epochs, 2000 muestras train, 500 val) debe caber en ~15-25 min total con GPU T4. Los resultados no son cientificos; solo validan que el pipeline funciona de punta a punta.

## 1. Drive + clonar repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!rm -rf /content/tfm_deteccion
!git clone https://github.com/manuelpalasanchez/tfm_deteccion.git
%cd /content/tfm_deteccion
!git pull

In [ ]:
!pip install -q -r requirements.txt

## 2. Extraccion reducida de datos

Solo 2 categorias de progan_train (car, horse) + progan_val completo. Muy pocos GB, extraccion en segundos.

In [ ]:
import subprocess, os

drive_zips = '/content/drive/MyDrive/cnndetection-datasets'
cnn_root   = '/content/cnndetection'
os.makedirs(f'{cnn_root}/progan_train', exist_ok=True)
os.makedirs(f'{cnn_root}/progan_val',   exist_ok=True)

cats_mini = ['car', 'horse']   # 2 categorias; cambiar a gusto
for cat in cats_mini:
    print(f'[train] extrayendo {cat}...')
    subprocess.run(['unzip', '-q', '-n', f'{drive_zips}/progan_train.zip',
                    f'{cat}/*', '-d', f'{cnn_root}/progan_train'], check=True)

print('[val] extrayendo progan_val completo...')
subprocess.run(['unzip', '-q', '-n', f'{drive_zips}/progan_val.zip',
                '-d', f'{cnn_root}/progan_val'], check=True)

print('\nListo. Estructura:')
!ls {cnn_root}
!ls {cnn_root}/progan_train
!du -sh {cnn_root}

## 3. Sanity check del dataset

In [ ]:
import sys
sys.path.insert(0, '/content/tfm_deteccion')
from data.cnndetection_dataset import CNNDetectionDataset
from data.transforms import get_eval_transforms

for split in ('train', 'val'):
    ds = CNNDetectionDataset(root='/content/cnndetection', split=split, transform=get_eval_transforms())
    print(f'{split}: {len(ds)} muestras, generadores={ds.generators}')

## 4. Train + evaluate de los 4 modelos

Los configs mini (`configs/mini_*.yaml`) usan:
- train: split=train de progan_train, max_samples=2000
- val: progan_val, max_samples=500
- eval_e1: progan_val, max_samples=500
- epochs=2, batch_size=32, num_workers=2, wandb=false
- output.base_dir = /content/drive/MyDrive/tfm-checkpoints-mini (persiste si se cae la sesion)

In [ ]:
!python scripts/run_all_mini.py

### Alternativa: lanzar modelo a modelo para iterar mas rapido

In [ ]:
modelo = 'resnet50'   # resnet50 | freqnet | vit | universalfakedetect
!python scripts/train.py --config configs/mini_{modelo}.yaml

In [ ]:
import glob

modelo = 'resnet50'
ckpts = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints-mini/{modelo}_*/checkpoint_best.pth'))
assert ckpts, f'Sin checkpoint para {modelo}'
ckpt = ckpts[-1]
print('checkpoint:', ckpt)
!python scripts/evaluate.py --config configs/mini_{modelo}.yaml --checkpoint "{ckpt}"

## 5. Resumen de metricas

In [ ]:
import glob, json
from pathlib import Path

for modelo in ['resnet50', 'freqnet', 'vit', 'universalfakedetect']:
    runs = sorted(glob.glob(f'/content/drive/MyDrive/tfm-checkpoints-mini/{modelo}_*'))
    if not runs:
        print(f'{modelo}: sin runs')
        continue
    last = Path(runs[-1])
    mpath = last / 'metrics.json'
    if not mpath.exists():
        print(f'{modelo}: {last.name} (sin metrics.json)')
        continue
    m = json.loads(mpath.read_text())
    for ronda, vals in m.items():
        print(f"{modelo:<22} {ronda:<8} AUC={vals['auc_roc']:.4f} AP={vals['average_precision']:.4f} Acc={vals['accuracy']:.4f}")